In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
import pickle

In [2]:
data = pd.read_csv('../../data/raw/data.csv')[['image_id','fold','label']]

In [3]:
score0 = pd.read_csv('../../score/version0.csv')
score1 = pd.read_csv('../../score/version4.csv') 
score2 = pd.read_csv('../../score/version6.csv') 
score3 = pd.read_csv('../../score/version5.csv') 
score4 = pd.read_csv('../../score/version1.csv') 

In [4]:
score0.columns = ['image_id','label'] + ['scr_v0_' + str(x) for x in range(5)]
score1.columns = ['image_id','label'] + ['scr_v1_' + str(x) for x in range(5)]
score2.columns = ['image_id','label'] + ['scr_v2_' + str(x) for x in range(5)]
score3.columns = ['image_id','label'] + ['scr_v3_' + str(x) for x in range(5)]
score4.columns = ['image_id','label'] + ['scr_v4_' + str(x) for x in range(5)]

In [5]:
data = data.merge(score0, on=['image_id','label'])
data = data.merge(score1, on=['image_id','label'])
data = data.merge(score2, on=['image_id','label'])
data = data.merge(score3, on=['image_id','label'])
data = data.merge(score4, on=['image_id','label'])

In [6]:
features = [x for x in data.columns if 'scr' in x]
baseline = [x for x in data.columns if 'scr_v3_' in x]

In [7]:
def buildModel(fold):
    train = data[data['fold'] != fold].copy().reset_index(drop=True)
    valid = data[data['fold'] == fold].copy().reset_index(drop=True)
    driver = valid[['image_id','fold','label']].copy()
    X_train, y_train = np.array(train[features]), np.array(train['label'])
    X_valid, y_valid = np.array(valid[features]), np.array(valid['label'])
    y_base = np.argmax(np.array(valid[baseline]), axis=1)
    model = LogisticRegression(max_iter=500, multi_class='ovr', C=0.2)
    model.fit(X_train, y_train)
    y_model = model.predict(X_valid)
    driver['baseline'] = y_base
    driver['blend'] = y_model
    driver['acc'] = (driver['blend'] == driver['label']).astype(int)
    print('Baseline:', np.mean(y_base == y_valid))
    print('Model:', np.mean(y_model == y_valid))
    pickle.dump(model, open('../../model/blend/blend_{}.pkl'.format(fold), 'wb'))
    return driver

In [8]:
blend_0 = buildModel(0)

Baseline: 0.8873831775700934
Model: 0.8943925233644859


In [9]:
blend_1 = buildModel(1)

Baseline: 0.8953271028037383
Model: 0.8990654205607477


In [10]:
blend_2 = buildModel(2)

Baseline: 0.8957700397289087
Model: 0.9055854171535406


In [11]:
blend_3 = buildModel(3)

Baseline: 0.89413414349147
Model: 0.9041832203785931


In [12]:
blend_4 = buildModel(4)

Baseline: 0.8997429305912596
Model: 0.9011451273662071


In [13]:
data = blend_0.append(blend_1).append(blend_2).append(blend_3).append(blend_4)

In [14]:
data['acc'].mean()

0.9008739542926578

In [15]:
data.groupby('label')['acc'].mean()

label
0    0.673413
1    0.812700
2    0.810981
3    0.975376
4    0.774544
Name: acc, dtype: float64